In [ ]:
!pip install deep-translator gTTS -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.


In [ ]:
from deep_translator import GoogleTranslator
from gtts import gTTS
import ipywidgets as widgets
from IPython.display import display, HTML, Audio, clear_output
import html

# Supported languages (naam -> code)
langs = GoogleTranslator().get_supported_languages(as_dict=True)
names = sorted(langs.keys())

source_options = [("Auto Detect", "auto")] + [(n.title(), langs[n]) for n in names]
target_options = [(n.title(), langs[n]) for n in names]

print(f"Total supported languages: {len(names)}")

Total supported languages: 133


In [ ]:
import time
from deep_translator import GoogleTranslator, MyMemoryTranslator
try:
    from langdetect import detect
except ImportError:
    # Fallback if langdetect is not installed
    def detect(text):
        return "en"

code_to_name = {v: k for k, v in langs.items()}
cache = {}  # same text dobara translate ho to request nahi jati

def chunk_text(text, size=450):
    """MyMemory ki limit (500 chars) ke liye text ko tukron mein todta hai."""
    parts, current = [], ""
    for sentence in text.replace("\n", " \n ").split(". "):
        if len(current) + len(sentence) + 2 > size:
            parts.append(current)
            current = ""
        current += sentence + ". "
    if current.strip():
        parts.append(current)
    return [p.strip() for p in parts if p.strip()]

def google_translate(text, source, target, retries=3):
    for attempt in range(retries):
        try:
            return GoogleTranslator(source=source, target=target).translate(text)
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 * (attempt + 1))   # 2s, 4s wait
            else:
                raise e

def mymemory_translate(text, source, target):
    if source == "auto":
        try:
            source = detect(text)               # e.g. 'en'
        except Exception:
            source = "en"
    src_name = code_to_name.get(source, "english")
    tgt_name = code_to_name.get(target, "urdu")
    out = [MyMemoryTranslator(source=src_name, target=tgt_name).translate(p)
           for p in chunk_text(text)]
    return " ".join(out)

def translate_text(text, source, target):
    """Pehle Google try karta hai, fail ho to MyMemory use karta hai."""
    if not text.strip():
        return "Please enter some text."

    key = (text, source, target)
    if key in cache:
        return cache[key]

    try:
        result = google_translate(text, source, target)
    except Exception:
        try:
            result = mymemory_translate(text, source, target)
        except Exception as e2:
            return f"Error: Dono services busy hain, thori der baad try karein. ({e2})"

    cache[key] = result
    return result

In [ ]:

text_input = widgets.Textarea(
    placeholder="Yahan text likhein...",
    description="Text:",
    layout=widgets.Layout(width="650px", height="120px")
)

source_dd = widgets.Dropdown(options=source_options, value="auto",
                             description="From:", layout=widgets.Layout(width="320px"))
target_dd = widgets.Dropdown(options=target_options, value="ur",
                             description="To:", layout=widgets.Layout(width="320px"))

translate_btn = widgets.Button(description="Translate", button_style="success", icon="language")
swap_btn = widgets.Button(description="Swap", icon="exchange")
speak_btn = widgets.Button(description="Speak", button_style="info", icon="volume-up")

output = widgets.Output()
last_translation = {"text": ""}

def on_translate(_):
    with output:
        clear_output()
        result = translate_text(text_input.value, source_dd.value, target_dd.value)
        last_translation["text"] = result
        safe = html.escape(result)
        display(HTML(f"""
        <div style="padding:12px;border:1px solid #4CAF50;border-radius:8px;
                    max-width:650px;font-size:18px;">
          <b>Translated Text:</b>
          <p id="result" style="white-space:pre-wrap;">{safe}</p>
          <button onclick="navigator.clipboard.writeText(document.getElementById('result').innerText);
                           this.innerText='Copied!';"
                  style="padding:6px 14px;cursor:pointer;">Copy</button>
        </div>
        """))

def on_swap(_):
    if source_dd.value == "auto":
        with output:
            clear_output()
            print("Auto Detect ke saath swap nahi ho sakta. Pehle source language select karein.")
        return
    source_dd.value, target_dd.value = target_dd.value, source_dd.value
    if last_translation["text"]:
        text_input.value = last_translation["text"]

def on_speak(_):
    with output:
        text = last_translation["text"]
        if not text or text.startswith("Error") or text.startswith("Please"):
            print("Pehle translate karein.")
            return
        try:
            tts = gTTS(text=text, lang=target_dd.value)
            tts.save("speech.mp3")
            display(Audio("speech.mp3", autoplay=True))
        except Exception as e:
            print(f"Is language ke liye speech available nahi: {e}")

translate_btn.on_click(on_translate)
swap_btn.on_click(on_swap)
speak_btn.on_click(on_speak)

display(HTML("<h2>🌐 Language Translation Tool</h2>"))
display(text_input,
        widgets.HBox([source_dd, target_dd]),
        widgets.HBox([translate_btn, swap_btn, speak_btn]),
        output)

Textarea(value='', description='Text:', layout=Layout(height='120px', width='650px'), placeholder='Yahan text …

Output()

In [ ]:
print(translate_text("Hello, how are you?", "en", "ur"))
print(translate_text("Artificial Intelligence is the future.", "auto", "fr"))

ہیلو آپ کیسے ہیں؟
